This notebook incrementally calculates profit estimates of performing skill based actions on an hourly basis and writes the data to  
Table xxx

In [22]:
import pyspark.sql.functions as sf
from time import time

Enriched Data

In [23]:
# Read data from 'runescape.02_silver.1h_prices_enriched'
df_1h_prices = spark.read.table("runescape.02_silver.1h_prices_enriched")
df_1h_prices = df_1h_prices.drop("highalch","limit","members")

In [24]:
# get skill input data
df_skill_inputs = spark.read.table("runescape.02_silver.skill_action_inputs_enriched")

In [ ]:
# TODO replace the code below with code that gets the most recent data from 'runescape.02_silver.1h_prices_enriched'
# instead of using the time window
# will need to make sure that the merge works when writing to gold layer.

In [ ]:
# filter data to only last 90 minutes (extra time allowed for any delays)
# job will run this notebook every hour
unix_time_window = time() - 5400 # 90 minutes
df_1h_prices = df_1h_prices.filter(f"time > {unix_time_window}")
# validate there is only 1 value for time in df_1h_prices
time_range = df_1h_prices.select("time").distinct().count()
if time_range > 1:
    raise Exception("There is more than 1 time present in df_1h_prices")
# df with the time for this data set.  This will be used later for the imputed data.
df_time = df_1h_prices.select(sf.first_value("time").alias("time"))
# check to make sure the time value is not null
if df_time.collect()[0][0] is None:
    raise Exception("The time in df_time is Null")

In [39]:
df_1h_prices.show()

+-----+----------+------------+------------------+-----------+-----------------+--------------------+
|   id|      time|avg1HourHigh|avg1HourHighVolume|avg1HourLow|avg1HourLowVolume|                name|
+-----+----------+------------+------------------+-----------+-----------------+--------------------+
|22275|1772636400|        NULL|              NULL|       NULL|             NULL|                NULL|
| 1133|1772636400|        NULL|              NULL|       NULL|             NULL|                NULL|
|22204|1772636400|        NULL|              NULL|       NULL|             NULL|                NULL|
| 6257|1772636400|        NULL|              NULL|       NULL|             NULL|                NULL|
|21117|1772636400|        NULL|              NULL|       NULL|             NULL|                NULL|
|11072|1772636400|        NULL|              NULL|       NULL|             NULL|                NULL|
| 1673|1772636400|        NULL|              NULL|       NULL|             NULL|  

Need to create a dataframe for all skill action outputs (and prices/ volumes) and thier respective inputs (and prices/ volumes)  
This dataframe also should have price data (and inputs) for items that did not have data in the last hour and instead use historic data from table 1h_prices_last_enriched

In [27]:
# get df with all distinct skill action output and input id's
# cross join with df_time 
# TODO consider how this is impacted by implementation of other skills
# TODO consider renaming df_outputs_time to something less dumb
df_outputs_time = df_skill_inputs\
    .select("id_output")\
    .union(df_skill_inputs\
    .select("id_input"))\
    .dropDuplicates()\
    .withColumnRenamed("id_output","id")\
    .crossJoin(df_time)
# join df_outputs_time with df_1h_prices to create df for profit
df_1h_prices = df_1h_prices.join(df_outputs_time, ["id", "time"], "right_outer")

Data imputation for missing/ filtered out data.  
Imputation method: Use the last known avg values based on hourly data in table 'runescape.02_silver.1h_prices_last_enriched'  
This approach has it's limiations and should be reconsidered.

In [28]:
# combine records with missing price data with data from 1h_prices_last to get imputed data for missing data
# get item time combination for missing data
df_items_times_missing = df_1h_prices.filter(sf.isnull("avg1HourHigh")).select("id","time")

In [29]:
# Read data from 'runescape.02_silver.1h_prices_last_enriched'
df_1h_prices_last = spark.read.table("runescape.02_silver.1h_prices_last_enriched")
# drop time column
df_1h_prices_last = df_1h_prices_last.drop("time", "highalch", "limit", "members")

In [30]:
# comnbine with data from 1h_prices_last
df_1h_prices_imputed = df_items_times_missing.join(df_1h_prices_last, "id")

In [31]:
# get real price data without Nulls
df_1h_prices_real = df_1h_prices.filter(sf.isnotnull("avg1HourHigh"))

# union 1h_price data to get complete data set
df_1h_prices_full = df_1h_prices_real.union(df_1h_prices_imputed)

Build table with profit calculations

In [32]:
#df_1h_skill_profit = df_1h_prices_full
# combine input data with price data
df_1h_prices_full_w_inputs = df_1h_prices_full.join(df_skill_inputs,
                                            df_1h_prices_full.id == df_skill_inputs.id_output,
                                            "left_outer")
df_1h_prices_full_w_inputs = df_1h_prices_full_w_inputs.drop('id','name','highalch','limit','members')
# rename columns to be associated with output data
df_1h_prices_full_w_inputs = df_1h_prices_full_w_inputs.withColumnsRenamed(
    {'avg1HourHigh':'avg_high_output',
     'avg1HourHighVolume':'avg_high_volume_output',
     'avg1HourLow':'avg_low_output',
     'avg1HourLowVolume':'avg_low_volume_output'}
)

In [40]:
df_1h_prices_full_w_inputs.show()

+----------+---------------+----------------------+--------------+---------------------+--------------------+---------+------------+--------+---------+--------+
|      time|avg_high_output|avg_high_volume_output|avg_low_output|avg_low_volume_output|              output|input_qty|       input|   skill|id_output|id_input|
+----------+---------------+----------------------+--------------+---------------------+--------------------+---------+------------+--------+---------+--------+
|1772636400|              3|                109174|             2|                 4229|                Vial|      1.0|Molten glass|crafting|      229|    1775|
|1772636400|             55|                 11535|            53|                12681|       Unpowered orb|      1.0|Molten glass|crafting|      567|    1775|
|1772636400|           1278|                  2507|          1262|                 3523|                NULL|     NULL|        NULL|    NULL|     NULL|    NULL|
|1772636400|            917|      

In [34]:
# join to get input price data
df_1h_prices_full_combined = df_1h_prices_full_w_inputs.join(df_1h_prices_full,
                                                                     [df_1h_prices_full_w_inputs.id_input == df_1h_prices_full.id,
                                                                      df_1h_prices_full_w_inputs.time == df_1h_prices_full.time],
                                                                     "left_outer").drop(df_1h_prices_full_w_inputs.time)

In [35]:
# calculate input costs
df_input_cost = df_1h_prices_full_combined.withColumns({'input_cost_low': sf.round(sf.col("avg1HourLow")*sf.col("input_qty"),1),
                                                            'input_cost_high': sf.round(sf.col("avg1HourHigh")*sf.col("input_qty"),1)})

In [36]:
df_input_total_cost = df_input_cost.groupBy("time","output", "skill").sum("input_cost_high", "input_cost_low")
df_input_total_cost = df_input_total_cost.withColumnsRenamed({"sum(input_cost_high)": "total_input_cost_high"
                                                              , "sum(input_cost_low)": "total_input_cost_low"})


In [37]:
tax = .98 #simplified tax for now, can be implemented as function
df_1h_profits = (df_1h_prices_full_combined.join(df_input_total_cost, ["time", "output", "skill"])\
    .drop("name")\
    # TODO rename input cols
    # calculate profit estimates without tax
    # TODO add tax and implement as function
    .withColumns({
        "profit_low_est": sf.round(tax*sf.col("avg_low_output") - sf.col("total_input_cost_high"),2),
        "profit_high_est": sf.round(tax*sf.col("avg_high_output") - sf.col("total_input_cost_low"),2)
    }))

In [41]:
df_1h_profits.show()

+----------+--------------------+--------+---------------+----------------------+--------------+---------------------+---------+--------------------+---------+--------+----+------------+------------------+-----------+-----------------+---------------------+--------------------+--------------+---------------+
|      time|              output|   skill|avg_high_output|avg_high_volume_output|avg_low_output|avg_low_volume_output|input_qty|               input|id_output|id_input|  id|avg1HourHigh|avg1HourHighVolume|avg1HourLow|avg1HourLowVolume|total_input_cost_high|total_input_cost_low|profit_low_est|profit_high_est|
+----------+--------------------+--------+---------------+----------------------+--------------+---------------------+---------+--------------------+---------+--------+----+------------+------------------+-----------+-----------------+---------------------+--------------------+--------------+---------------+
|1772636400|                Vial|crafting|              3|            